# 05 · Model Comparison & Champion
Pick best model and export artifacts.


In [ ]:
from pathlib import Path
import json, shutil, joblib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

DATA_DIR = Path("../../data_pipeline/data/processed")
MODEL_DIR = Path("../models")
ALT_MODEL = Path("../../apps/backend/ml_artifacts")
REPORT_DIR = Path("../../docs/reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load data with fallbacks
fpath = MODEL_DIR / "feature_columns.json"
if not fpath.exists():
    fpath = ALT_MODEL / "feature_columns.json"
with open(fpath) as f:
    feature_cols = json.load(f)

if (DATA_DIR / "X_test.csv").exists():
    X_test = pd.read_csv(DATA_DIR / "X_test.csv")[feature_cols]
    y_test = pd.read_csv(DATA_DIR / "y_test.csv")["target_risk"].astype(int)
else:
    tm = pd.read_csv(DATA_DIR / "test_matrix.csv")
    X_test = tm[feature_cols]
    y_test = tm["target_risk"].astype(int)

model_names = ["Logistic Regression","Random Forest","XGBoost","LightGBM"]
models = {}
rows = []
for name in model_names:
    slug = name.lower().replace(" ","_")
    p = MODEL_DIR / f"{slug}.joblib"
    if not p.exists():
        print(f"Missing {p}, skip {name}")
        continue
    m = joblib.load(p)
    models[name] = m
    pred = m.predict(X_test)
    prob = m.predict_proba(X_test)[:,1]
    rows.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, pred),4),
        "Precision": round(precision_score(y_test, pred, zero_division=0),4),
        "Recall": round(recall_score(y_test, pred, zero_division=0),4),
        "F1": round(f1_score(y_test, pred, zero_division=0),4),
        "ROC-AUC": round(roc_auc_score(y_test, prob),4),
    })

comp_df = pd.DataFrame(rows).sort_values(["Recall","F1","ROC-AUC"], ascending=False).reset_index(drop=True)
display(comp_df)
print(comp_df.to_string(index=False))


In [ ]:
# Bar chart comparison
metrics = ["Accuracy","Precision","Recall","F1","ROC-AUC"]
x = np.arange(len(metrics))
width = 0.2
colors = ["#1b5e20","#2e7d32","#66bb6a","#8d6e63"]
fig, ax = plt.subplots(figsize=(12,6))
for i, row in comp_df.iterrows():
    ax.bar(x + i*width, [row[m] for m in metrics], width, label=row["Model"], color=colors[i%len(colors)])
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(metrics)
ax.set_ylim(0,1.1)
ax.set_title("4-Model Comparison (2024-2025 Test)")
ax.set_ylabel("Score")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Champion selection and export
champion_name = comp_df.iloc[0]["Model"]
champion = models[champion_name]
print(f"Champion: {champion_name}")
joblib.dump(champion, MODEL_DIR / "champion_model.joblib")
# Also save as xgboost if champion is XGBoost for backend compat
if "XGBoost" in models:
    joblib.dump(models["XGBoost"], MODEL_DIR / "xgboost_agrishield_v1.joblib")
comp_df.to_csv(MODEL_DIR / "model_comparison_2015_2025.csv", index=False)

# Sync to backend if exists
if ALT_MODEL.exists():
    for fname in ["champion_model.joblib","xgboost_agrishield_v1.joblib","scaler.joblib","feature_columns.json","county_ndvi_stats.csv"]:
        src = MODEL_DIR / fname
        if src.exists():
            shutil.copy2(src, ALT_MODEL / fname)
    print(f"Synced to {ALT_MODEL}")

# Simple report
md = f"# ML Comparison Report\n\nChampion: **{champion_name}**\n\n{comp_df.to_markdown(index=False)}\n"
(REPORT_DIR / "ML_Model_Comparison_Report_2015_2025.md").write_text(md)
print(f"Report saved to {REPORT_DIR}")
